# QA Validator

### Setup & imports

In [ ]:
# Setup & imports
import pandas as pd
import json
import os

from openai import AzureOpenAI

In [ ]:
# Computation
endpoint = os.getenv("ENDPOINT_URL", "")
deployment = os.getenv("DEPLOYMENT_NAME", "gpt-4.1")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY", "")  

# Initialize Azure OpenAI client with key-based authentication",
client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=subscription_key,
    api_version="2025-01-01-preview",
)

In [16]:

#prompt builder

def build_prompt(passages, system_prompt):

    user_content = []

    for passage in passages:
        user_content.append({"type": "text",
                             "text": passage})

    chat_prompt = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": system_prompt
                }
            ]
        },
        {
            "role": "user",
            "content": user_content
        }
    ]
    return chat_prompt


In [17]:
# Modeling & evaluation
def openai_api(lines, system_prompt):
    # Include speech result if speech is enabled
    messages = build_prompt(lines, system_prompt)
    
    # Generate the completion
    completion = client.chat.completions.create(
        model=deployment,
        messages=messages,
        max_tokens=2000,
        temperature=0.7,
        top_p=0.95,
        frequency_penalty=0,
        presence_penalty=0,
        stop=None,
        stream=False
    )
    
    return completion.choices[0].message.content

In [ ]:
# Load JSON file as a Python object  
with open("./questionanswers_complete.json") as f:  
    data = json.load(f)  
    
qa = pd.DataFrame(data)  
qa = qa.astype({  
    "file": "string",   
})  

In [19]:
# Computation
def find_file(directory, filename):  
    for root, dirs, files in os.walk(directory):  
        if filename in files:  
            return os.path.join(root, filename)  
    return None  

In [20]:
# Data loading
generated_answers = []

system_prompt = '''Analyze the provided question and accompanying information to generate a concise one-sentence answer that addresses the core inquiry directly and accurately.

# Steps

1. **Understand Context**: Fully interpret both the question and the provided information to establish the core topic and expected output.
2. **Extract Key Points**: Identify relevant details within the information that directly inform the question.
3. **Analyze Information**: Use logical reasoning to synthesize an understanding of how the provided details answer the question.
4. **Craft the Sentence**: Based on the analysis, generate a clear and concise one-sentence answer that directly addresses the question.

# Output Format

A single, clear sentence providing a direct answer to the question.

# Notes

- Ensure that the one-sentence answer includes all critical details needed for clarity.
- Avoid vague or overly broad responses; be specific and accurate.
- If the information provided contains ambiguity, acknowledge it within the sentence as clearly as possible.'''

for index, row in qa.iterrows():
    directory_to_search = "/path/to/directory"  
    file_to_find = "example.txt"  
    
    result = find_file('/home/ec2-user/repos/msc/conversations', row['file'] + '.txt')  
    if result:
        with open(result, "r") as file:  
            file_contents = file.read() 
            generated_answers.append(openai_api([row['question'], file_contents], system_prompt))
    else:  
        print("File not found")  

In [21]:
# Computation
qa['generated_answers_gpt4.1'] = generated_answers

In [22]:
# Computation
qa.to_json('questionanswers_complete_generated_v2.json', orient='records', indent=4)

In [24]:
# Modeling & evaluation
conclusion = []

system_prompt = '''Determine if the provided "expected" and "actual" answers match or differ, allowing for some variation and differences in detail between the answers.

Consider the answers to match if their meaning is the same, even if the wording, phrasing, or specific details vary. Identify cases where differences in meaning or intent occur. Provide reasoning for the match or mismatch determination. When evaluating, take into account allowances for acceptable variations but flag meaningful discrepancies.

# Steps

1. Compare the "expected" answer and the "actual" answer.
2. Identify whether the key meaning and intent are preserved across both answers.
3. Note any differences in detail, language, or structure and assess whether they materially affect the overall meaning.
4. Determine if the differences are minor or significant to decide if the answers "match" or "differ."
5. Provide reasoning to justify whether the answers match or differ.

# Output Format

The output should be structured as follows:

```json
{
  "match": [true/false],
  "reasoning": "[Provide detailed reasoning explaining why the answers match or differ, referencing key points in the 'expected' and 'actual' answers.]"
}
```

# Examples

**Example 1:**

- **Expected:** "Cats are popular pets because they are independent and require less maintenance."
- **Actual:** "Cats are favored by many as they are self-sufficient and low-maintenance."

**Output:**

```json
{
  "match": true,
  "reasoning": "Both answers convey the same meaning: cats are desirable pets because they are independent and low-maintenance. The differences in wording do not affect the overall intent or key points."
}
```

**Example 2:**

- **Expected:** "Photosynthesis is the process by which plants convert sunlight into energy."
- **Actual:** "Photosynthesis is when plants use sunlight to grow and make food."

**Output:**

```json
{
  "match": true,
  "reasoning": "Both describe photosynthesis as plants using sunlight to create energy. While 'make food' is less precise than 'convert sunlight into energy,' the general meaning aligns sufficiently to consider them a match."
}
```

**Example 3:**

- **Expected:** "The capital of France is Paris."
- **Actual:** "The largest city in France is Paris."

**Output:**

```json
{
  "match": false,
  "reasoning": "The expected answer identifies Paris as the capital of France, while the actual answer focuses on its size. The answers differ in meaning, as being the capital and being the largest city are not necessarily the same."
}
``` 

**Example 4:**

- **Expected:** "World War I started in 1914 due to various political and social tensions."
- **Actual:** "World War I began in 1914, triggered by the assassination of Archduke Franz Ferdinand."

**Output:**

```json
{
  "match": true,
  "reasoning": "Although the answers reference different specific causes, both agree on the year and provide complementary explanations for the start of World War I. They align sufficiently to be considered a match."
}
```

# Notes

- Allow for minor rewordings, synonymous phrasing, or omitted secondary details without altering the core meaning.
- Significant differences in accuracy, key details, or intent should result in the determination of a mismatch.'''

for index, row in qa.iterrows():
    conclusion.append(openai_api(['expected answer: ' + row['answer'], 'actual answer: ' + row['generated_answers_gpt4.1']], system_prompt))

In [25]:
# Computation
qa['evaluation_gpt4.1'] = conclusion

In [ ]:
# Load JSON file as a Python object  
with open("./questionanswers_complete_generated_evaluated.json") as f:  
    data = json.load(f)  
    
qa = pd.DataFrame(data)  
qa = qa.astype({  
    "file": "string",   
})  

In [28]:
# Computation
subset_df = qa[qa['evaluation_gpt4.1'].str.contains('"match": false', case=False, na=False)]  

In [29]:
# Computation
subset_df.to_json('questionanswers_complete_generated_no_match_v2.json', orient='records', indent=4)